# TP A — Collecter des données : API, crawling et scraping

## Où vous allez
- **Ce que vous allez produire :** un fichier `corpus.csv` de quelques centaines de textes classés par thème (4 thèmes que **vous** choisissez), collectés par API, crawling et scraping
- **Ce que vous devrez rendre :** le notebook complété (contrôles verts), le fichier `corpus.csv`, et vos réponses aux questions ✍️
- **Et ensuite :** au TP B, vous entraînerez un réseau de neurones sur ce corpus pour retrouver le thème d'un texte

## Ce que vous allez apprendre
- Faire la différence entre une **API**, le **crawling** et le **scraping**, et savoir quand utiliser chacun
- Collecter des textes de façon **automatisée** (pause, nouvel essai, `User-Agent`, `robots.txt`)
- Gérer la **pagination** d'une API et **parcourir** un arbre de catégories
- **Trouver vous-même** les catégories à collecter (par l'API et par le navigateur), puis **choisir vos thèmes**
- Produire un **fichier propre** (`corpus.csv`) qui servira à entraîner un réseau de neurones dans les TP suivants


## Boîte à outils
Noms de fonctions et de classes utiles. **Aucun exemple d'appel n'est donné** : cherchez dans la documentation comment les utiliser.

`requests.get`  ·  `allcategories`  ·  `BeautifulSoup`  ·  `select`  ·  `select_one`  ·  `get_text`  ·  `RobotFileParser`  ·  `time.sleep`  ·  `pd.DataFrame`  ·  `drop_duplicates`  ·  `to_csv`

## Comment travailler
- Les cellules **déjà écrites** sont à lire et à exécuter.
- Les cellules **Étape** sont à **compléter** (un `___` ou un `# TODO`), à partir de la consigne. Chaque étape indique **ce que vous devez obtenir**.
- Après chaque étape, une cellule **✅ Contrôle** vérifie votre résultat. Respectez les **noms de variables** demandés.
- Les questions **✍️** demandent une réponse écrite : c'est là que vous montrez que vous avez compris.
- Bloqué ? Relisez l'indice, puis cherchez dans la documentation. Un message d'erreur se lit : la dernière ligne dit souvent ce qui ne va pas.

In [1]:
import os, re, time
import requests, pandas as pd
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

BASE = os.environ.get("WIKI_BASE", "https://en.wikipedia.org")     # le site visé
PAUSE = float(os.environ.get("WIKI_PAUSE", "1.0"))                 # secondes d'attente entre deux requêtes
ENTETES = {"User-Agent": "TPFormation/1.0 (frederic.perret@uni-simplon.fr) python-requests"}   # ⚠️ remplacez par VOTRE contact

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## 🧭 Trois façons de récupérer des données sur le web
| Technique | En une phrase | Ce qu'on reçoit | Quand l'utiliser |
|---|---|---|---|
| **API** | Une porte d'entrée **officielle** prévue pour les programmes | Des données **déjà structurées** (JSON) | Dès qu'elle existe : c'est le plus propre et le plus stable |
| **Crawling** | Un programme **parcourt les pages en suivant les liens** (comme le robot de Google) | La **liste** des pages à traiter | Pour **découvrir** des pages quand on n'a pas la liste |
| **Scraping** | Un programme **lit le HTML** d'une page et en **extrait des champs précis** | Des données **extraites** d'une page faite pour les humains | Quand aucune API ne donne ce qu'on veut |

**Dans ce TP, on utilise les trois sur Wikipédia** pour constituer un corpus de textes classés par thème (astronomie, cuisine, peinture, football). Ce corpus servira ensuite au deep learning.

## Les règles du jeu (non négociables)
- **Se présenter** : Wikimedia exige un `User-Agent` descriptif **avec un contact**. Sans ça, vous serez bloqué.
- **Ne pas harceler le serveur** : une requête à la fois, avec une **pause**. Si le serveur dit « trop de requêtes » (code 429), on **ralentit**.
- **Lire `robots.txt`** : le site y dit ce que les robots ont le droit de visiter.
- **Respecter la licence** : le texte de Wikipédia est sous licence CC BY-SA. Pour un usage de formation, on cite la source.


## Étape 1/15 — Une fonction de requête « polie »
Toutes nos requêtes passeront par cette fonction. Elle **attend** avant chaque requête, **se présente** avec le `User-Agent`, et **réessaie** en ralentissant si le serveur répond « trop de requêtes » (429) ou « indisponible » (503).

In [2]:
def get(url, params=None, essais=3):
    for n in range(essais):
        time.sleep(PAUSE)                                   # pause : on ne harcèle pas le serveur
        reponse = requests.get(url, params=params, headers=ENTETES, timeout=20)
        if reponse.status_code == 200:
            return reponse
        if reponse.status_code in (429, 503):               # « trop de requêtes » : on attend plus longtemps
            time.sleep(2 ** (n + 1))
            continue
        reponse.raise_for_status()
    raise RuntimeError("Échec après " + str(essais) + " essais : " + url)

## Étape 2/15 — Lire le `robots.txt`
Avant d'explorer un site, on lit ses règles. `can_fetch` répond : « mon robot a-t-il le droit de visiter cette adresse ? ».

In [3]:
robots = RobotFileParser()
robots.parse(get(BASE + "/robots.txt").text.splitlines())
print("Visiter /wiki/Main_Page autorisé :", robots.can_fetch(ENTETES["User-Agent"], BASE + "/wiki/Main_Page"))
print("Visiter /wiki/Special:Search autorisé :", robots.can_fetch(ENTETES["User-Agent"], BASE + "/wiki/Special:Search"))

Visiter /wiki/Main_Page autorisé : True
Visiter /wiki/Special:Search autorisé : False


### ✍️ Question
**Pourquoi un programme de collecte doit-il lire `robots.txt` et se présenter avec un `User-Agent` ? Qu'arrive-t-il sinon ?**

*Votre réponse (2 à 4 lignes) :*

> 

Le fichier robots.txt définit les règles d'accès accordés aux agents par les responsables des sites. Ces règles ne bloquent pas les agents qui y contreviennent, mais définissent une utilisation respectueuse du web.

Si l'on ne veut pas déroger à ce comportement respectueux, il faut suivre ces règles, en commençant par indiquer quel agent est utilisé.

Certains sites se protègent contre les accès sans agent identifié et bloquent l'accès. Un accès non respectueux est tracé sur le site concerné avec l'IP appelant, la réputation de l'organisme associé à cet IP peut en pâtir.

## Étape 3/15 — Partie 1 — L'API : lister les membres d'une catégorie
L'API de Wikipédia renvoie les membres d'une catégorie en JSON. Elle n'en donne qu'**une page à la fois** (50 maximum) : tant que la réponse contient `continue`, il faut **redemander la suite** avec `cmcontinue`. C'est la **pagination**.

In [4]:
def membres_api(categorie, type_="page", maximum=200):
    params = {"action": "query", "list": "categorymembers", "cmtitle": "Category:" + categorie,
              "cmtype": type_, "cmlimit": 50, "format": "json"}
    titres = []
    while len(titres) < maximum:
        donnees = get(BASE + "/w/api.php", params).json()
        titres += [m["title"] for m in donnees["query"]["categorymembers"]]
        if "continue" not in donnees:                       # plus de page suivante
            break
        params["cmcontinue"] = donnees["continue"]["cmcontinue"]
    return titres[:maximum]

## Étape 4/15 — Trouver les catégories : à vous de chercher
Wikipédia range ses articles dans des **catégories**. Pour collecter un thème, il faut d'abord **connaître le nom exact** de sa catégorie : **personne ne vous le donne**, il faut le chercher. Deux façons :

- **Dans le navigateur** : ouvrez un article du thème et regardez en **bas de page** la liste de ses catégories ; cliquez sur l'une d'elles, son nom est dans l'adresse (`.../wiki/Category:Nom`).
- **Avec l'API** : on cherche les catégories **dont le nom commence par** un début de mot (`list=allcategories`, paramètre `acprefix`). La fonction ci-dessous le fait.

⚠️ Les noms sont **en anglais**, la **première lettre est une majuscule**, et la recherche porte sur le **début** du nom (un début comme `Geo` trouve les catégories dont le nom **commence** par `Geo`, mais pas celles qui contiennent `geo` au milieu du nom). Le nombre renvoyé est le nombre d'éléments de la catégorie : zéro n'a aucun intérêt. Les noms contiennent des **espaces** (`Association football`) alors que les adresses utilisent des **tirets bas** : notre code gère les deux.

In [5]:
def chercher_categories(debut, maximum=10):
    params = {"action": "query", "list": "allcategories", "acprefix": debut, "acmin": 1, "acprop": "size",
              "aclimit": maximum, "format": "json", "formatversion": 2}
    donnees = get(BASE + "/w/api.php", params).json()
    return [(c.get("category") or c.get("*"), c.get("size")) for c in donnees["query"]["allcategories"]]

## Étape 5/15 — Choisir vos thèmes et trouver leurs catégories
On constitue un corpus de **4 thèmes** très différents les uns des autres (sinon le réseau aura du mal à les distinguer). Thèmes conseillés : **astronomie, cuisine, peinture, football**. Vous pouvez en choisir d'autres, du moment qu'ils sont bien distincts.

**À faire :** 1) Pour chaque thème, **cherchez le nom de la catégorie** avec `chercher_categories(...)` (essayez plusieurs débuts de mot, en anglais, avec une majuscule) et/ou dans votre navigateur. 2) Construisez le dictionnaire **`CATEGORIES`** : clé = le thème en français, valeur = le **nom exact** de la catégorie Wikipédia. 3) Vérifiez qu'elles existent et ne sont pas vides.

**Ce que vous devez obtenir :** un dictionnaire `CATEGORIES` de 3 à 4 thèmes, avec le **nom exact** de chaque catégorie, dont l'existence est vérifiée par l'API.

> 💡 **Indice** : Essayez d'abord `print(chercher_categories("Geo"))` pour voir le format de la réponse, puis adaptez le début de mot à vos thèmes. Un nom en deux mots s'écrit avec une espace : `"Association football"`. Si une recherche ne renvoie rien, essayez un début plus court ou un autre mot.

In [120]:
print(chercher_categories("Artificial_intelligence"))     # exemple : adaptez le début de mot à vos thèmes

# DONE : {"thème en français": "Nom exact de la catégorie"}, pour 4 thèmes
CATEGORIES = {'IA': 'Artificial intelligence',
              'Dérive climatique': 'Climate change',
              'Voiture hydrogène': 'Hydrogen cars',
              'Chorégraphes': 'Ballet choreographers'}


[('Artificial intelligence', 319), ('Artificial intelligence articles needing expert attention', 6), ('Artificial intelligence associations', 34), ('Artificial intelligence by country', 9), ('Artificial intelligence by decade', 6), ('Artificial intelligence by year', 29), ('Artificial intelligence characters in video games', 22), ('Artificial intelligence companies', 177), ('Artificial intelligence competitions', 26), ('Artificial intelligence conferences', 41)]


In [101]:
# ✅ Contrôle (ne pas modifier)
controle(len(CATEGORIES) >= 3 and all(len(membres_api(n, 'page')) + len(membres_api(n, 'subcat')) > 0 for n in CATEGORIES.values()), '%d catégories valides : %s' % (len(CATEGORIES), ', '.join(CATEGORIES.values())), 'Attendu : au moins 3 catégories qui existent et ne sont pas vides')

✅ 4 catégories valides : Artificial intelligence, Climate change, Hydrogen cars, Choreographers


### ✍️ Question
**Comment avez-vous trouvé le nom exact de chaque catégorie ? Que faire si `chercher_categories` ne renvoie rien pour un thème ?**

*Votre réponse (2 à 4 lignes) :*

> 

En testant plusieurs mots en anglais avec chercher_categories, on obtient plusieurs noms exactes de catégories avec les sujets intéressants.

Pour vérifier, la recherche du nom de la catégorie sur le site permet de s'assurer des thèmes abordés dans la catégorie.

## Étape 6/15 — Observer une réponse brute de l'API
Avant de traiter des données, on **regarde** à quoi elles ressemblent. Une réponse d'API contient un **code de statut**, un **type de contenu** et un **corps** au format JSON.

**À faire :** Pour votre première catégorie (`categorie_test`), faites **une seule requête** à l'API en demandant 3 résultats. Affichez le **code de statut**, le **type de contenu** (en-tête `Content-Type`), les **clés du JSON** et le **premier membre** renvoyé.

**Ce que vous devez obtenir :** le statut, le type de contenu et les clés d'une vraie réponse d'API, et son premier membre.

> 💡 **Indice** : `reponse.status_code`, `reponse.headers["Content-Type"]`, `reponse.json()` (un dictionnaire), puis `.keys()`.

In [102]:
categorie_test = list(CATEGORIES.values())[0]
params = {"action": "query", "list": "categorymembers", "cmtitle": "Category:" + categorie_test, "cmlimit": 3, "format": "json"}
reponse = get(BASE + "/w/api.php", params)
# DONE : afficher le code de statut, le type de contenu, les clés du JSON et le premier membre
print(f"Statut : {reponse.status_code}, type de contenu : {reponse.headers['Content-Type']}, clés du JSON : {list(reponse.json().keys())}, premier membre : {next(iter(reponse.json().values()))}")


Statut : 200, type de contenu : application/json; charset=utf-8, clés du JSON : ['batchcomplete', 'continue', 'query'], premier membre : 


In [103]:
# ✅ Contrôle (ne pas modifier)
controle(reponse.status_code == 200 and 'query' in reponse.json(), 'Réponse reçue : statut %d, clés %s' % (reponse.status_code, list(reponse.json().keys())), 'Attendu une réponse 200 contenant la clé query')

✅ Réponse reçue : statut 200, clés ['batchcomplete', 'continue', 'query']


## Étape 7/15 — Appeler l'API sur votre première catégorie
On demande à l'API deux listes pour **votre première catégorie** : les **articles** directs (`type_="page"`) et les **sous-catégories** (`type_="subcat"`).

**À faire :** Appelez `membres_api` pour obtenir `articles_directs` et `sous_categories`, puis affichez leur nombre et les 5 premiers éléments de chaque liste.

**Ce que vous devez obtenir :** deux listes : les **articles directs** et les **sous-catégories** de votre première catégorie.

> 💡 **Indice** : `membres_api(categorie_test, "page")` et `membres_api(categorie_test, "subcat")`.

In [122]:
indice = 0
categorie_test = list(CATEGORIES.values())[indice]     # votre première catégorie
# DONE : appeler membres_api deux fois et afficher les résultats
articles_directs = membres_api(categorie_test, type_="page", maximum=20)
if len(articles_directs) > 0:
    print(f"Liste des articles de la catégorie {list(CATEGORIES.keys())[indice]} :")
    for art in articles_directs:
        if len(art.split(':')) == 1 or art.split(':')[0].lower().find("talk")<0:
            print(f"\t{BASE}/wiki/{art.replace(' ','_')}")
sous_categories = membres_api(categorie_test, type_="subcat", maximum=10)
if len(sous_categories) > 0:
    print(f"Liste des sous-catégories de la catégorie {list(CATEGORIES.keys())[indice]} :")
    for sscat in sous_categories:
        sous_cat = sscat.split(':')[1]
        print (f"\tArticles de la sous-catégorie {sous_cat} :")
        for art in membres_api(sous_cat, type_="page", maximum=20):
            if len(art.split(':')) == 1 or art.split(':')[0].lower().find("talk")<0:
                print(f"\t\t{BASE}/wiki/{art.replace(' ','_')}")


Liste des articles de la catégorie IA :
	https://en.wikipedia.org/wiki/Artificial_intelligence
	https://en.wikipedia.org/wiki/List_of_artificial_intelligence_journals
	https://en.wikipedia.org/wiki/2025–present_global_memory_supply_shortage
	https://en.wikipedia.org/wiki/The_2028_Global_Intelligence_Crisis
	https://en.wikipedia.org/wiki/Actor-critic_algorithm
	https://en.wikipedia.org/wiki/Admissible_heuristic
	https://en.wikipedia.org/wiki/Agent_experience
	https://en.wikipedia.org/wiki/Agent_harness
	https://en.wikipedia.org/wiki/Agent_verification
	https://en.wikipedia.org/wiki/Agentive_logic
	https://en.wikipedia.org/wiki/.ai
	https://en.wikipedia.org/wiki/AI_addiction
	https://en.wikipedia.org/wiki/AI_agent
	https://en.wikipedia.org/wiki/AI_alignment
	https://en.wikipedia.org/wiki/AI_anthropomorphism
	https://en.wikipedia.org/wiki/AI_browser
	https://en.wikipedia.org/wiki/AI_build-out_financing
	https://en.wikipedia.org/wiki/The_AI_Con
	https://en.wikipedia.org/wiki/AI_data_center

In [109]:
# ✅ Contrôle (ne pas modifier)
controle(len(articles_directs) + len(sous_categories) > 0, '%d articles directs et %d sous-catégories trouvés' % (len(articles_directs), len(sous_categories)), 'Attendu au moins un article ou une sous-catégorie')

✅ 20 articles directs et 10 sous-catégories trouvés


### ✍️ Question
**Il y a peu d'articles directement dans la catégorie et beaucoup de sous-catégories. Comment obtenir beaucoup plus de textes ?**

*Votre réponse (2 à 4 lignes) :*

> 

Déjà en affichant les articles de chaque sous-catégorie, on obtient beaucoup plus de résultats.

On peut donc balayer l'arborescence catégorie -> sous-catégorie jusqu'à une profondeur définie pour obtenir une liste significative d'articles.

## Étape 8/15 — Partie 2 — Le crawling : parcourir l'arbre des catégories
Ici on **suit les liens** dans le HTML : on lit une catégorie, on note ses articles, puis on va visiter ses sous-catégories. `vus` évite de visiter deux fois la même page, et `profondeur` limite jusqu'où on descend. Les sélecteurs `#mw-pages` et `#mw-subcategories` désignent les blocs HTML qui contiennent les listes.

In [112]:
def lire_categorie(nom):
    page = BeautifulSoup(get(BASE + "/wiki/Category:" + nom.replace(" ", "_")).text, "html.parser")
    sous = [a["href"].split("/wiki/Category:")[1] for a in page.select("#mw-subcategories a[href^='/wiki/Category:']")]
    articles = [a["href"] for a in page.select("#mw-pages li a") if ":" not in a["href"].split("/wiki/")[-1]]
    return sous, articles

def explorer(categorie, profondeur=1, maximum=120):
    a_voir, vus, trouves = [(categorie.replace(" ", "_"), 0)], set(), []
    while a_voir and len(trouves) < maximum:
        nom, niveau = a_voir.pop(0)
        if nom in vus:
            continue
        vus.add(nom)
        sous, articles = lire_categorie(nom)
        trouves += [a for a in articles if a not in trouves]
        if niveau < profondeur:
            a_voir += [(s, niveau + 1) for s in sous]
    return trouves[:maximum]

## Étape 9/15 — Mesurer l'effet de la profondeur
Plus on descend dans l'arbre, plus on trouve d'articles, mais plus ça prend de temps (une requête par catégorie visitée).

**À faire :** Pour `categorie_test`, appelez `explorer` avec `profondeur=0`, puis `profondeur=1` (avec `maximum=60`) et affichez le nombre d'articles trouvés dans chaque cas. Stockez les résultats dans `n_prof0` et `n_prof1`.

**Ce que vous devez obtenir :** deux nombres, `n_prof0` et `n_prof1`, qui montrent ce que le crawler gagne en descendant dans l'arbre.

> 💡 **Indice** : `len(explorer(categorie_test, profondeur=0, maximum=60))`.

In [ ]:
# DONE : calculer n_prof0 et n_prof1
indice = 3
categorie_test = list(CATEGORIES.values())[indice]     # votre première catégorie
n_prof0 = len(explorer(categorie_test, profondeur=0, maximum=200))
n_prof1 = len(explorer(categorie_test, profondeur=1, maximum=200))
print(f"Catégorie {categorie_test} : n_prof0 = {n_prof0}, n_prof1 = {n_prof1}")


Catégorie Ballet choreographers : n_prof0 = 172, n_prof1 = 200


In [126]:
# ✅ Contrôle (ne pas modifier)
controle(n_prof1 >= n_prof0 >= 0, 'Profondeur 1 : %d articles contre %d à la profondeur 0' % (n_prof1, n_prof0), 'Attendu : au moins autant d\'articles à la profondeur 1 qu\'à la profondeur 0')

✅ Profondeur 1 : 200 articles contre 172 à la profondeur 0


## Étape 10/15 — Observer le HTML brut d'un article
Une page web n'est que du **texte** : du HTML. Avant de le scraper, on le **regarde**, pour voir ce qu'on va devoir traverser.

**À faire :** Récupérez 3 liens d'articles de `categorie_test` (`explorer`, profondeur 1, `maximum=3`), téléchargez la page du **premier** avec `get(BASE + lien)`, et stockez son texte dans `html`. Affichez la **taille** de la page en caractères, ses **300 premiers caractères** et le **nombre de balises `<p`** qu'elle contient.

**Ce que vous devez obtenir :** la taille et le début du HTML d'une page, et le nombre de paragraphes qu'elle contient.

> 💡 **Indice** : `get(...).text` donne le HTML sous forme de texte. `html.count("<p")` compte les occurrences.

In [ ]:
indice = 0
categorie_test = list(CATEGORIES.values())[indice]     # votre première catégorie
liens_obs = explorer(categorie_test, profondeur=1, maximum=3)
lien = liens_obs[0]
# DONE : télécharger la page dans html, puis afficher sa taille, ses 300 premiers caractères et le nombre de balises <p
html = get(BASE + lien).text
print(f"Page {BASE + lien} :\nTaille de l'html : {len(html)}, nombre de paragraphes : {html.count('<p')}")
print(html[:300])


Page https://en.wikipedia.org/wiki/Artificial_intelligence :
Taille de l'html : 2257955, nombre de paragraphes : 188
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disab


In [131]:
# ✅ Contrôle (ne pas modifier)
controle(len(html) > 500 and html.count('<p') > 0, 'Page lue : %d caractères, %d balises <p' % (len(html), html.count('<p')), 'Attendu : une page de plus de 500 caractères qui contient des paragraphes')

✅ Page lue : 2257955 caractères, 188 balises <p


## Étape 11/15 — Partie 3 — Le scraping : extraire le texte d'un article
On lit maintenant le HTML d'**un article** et on en extrait deux champs : le **titre** et le **premier vrai paragraphe**. Les premiers paragraphes sont parfois vides ; on prend donc le premier qui fait plus de 150 caractères, et on retire les renvois de notes comme `[1]`.

⚠️ **Piège réel** : sur une page **protégée** (cadenas), Wikipédia place d'abord un autre bloc `mw-parser-output` pour l'icône, **avant** le vrai contenu. Un sélecteur trop large tombe sur ce bloc vide et ne trouve aucun paragraphe. On cible donc `#mw-content-text`, le vrai contenu.

In [132]:
def scraper_article(lien):
    page = BeautifulSoup(get(BASE + lien).text, "html.parser")
    titre = page.select_one("#firstHeading").get_text(strip=True)
    corps = page.select_one("#mw-content-text div.mw-parser-output")     # le VRAI contenu (pas le cadenas des pages protégées)
    if corps is None:
        return None
    for p in corps.find_all("p"):
        if p.find_parent("table"):                                      # on ignore les paragraphes des tableaux et infoboxes
            continue
        texte = re.sub(r"\[[^\]]*\]", "", p.get_text(" ", strip=True))   # retire les renvois [1], [a], [citation needed]
        if len(texte) > 150:
            return {"titre": titre, "texte": texte, "lien": lien}
    return None

## Étape 12/15 — Scraper un article
Essayons sur le premier article trouvé à l'étape précédente.

**À faire :** Récupérez la liste des articles de `categorie_test` (profondeur 1, `maximum=10`) dans `liens`, scrapez le **premier** avec `scraper_article` dans `fiche`, puis affichez son titre et ses 200 premiers caractères.

**Ce que vous devez obtenir :** un dictionnaire `fiche` contenant le titre et le premier vrai paragraphe d'un article.

> 💡 **Indice** : `liens = explorer(...)` puis `fiche = scraper_article(liens[0])`.

In [ ]:
# DONE : liens, fiche, affichage
indice = 0
categorie_test = list(CATEGORIES.values())[indice]     # votre première catégorie
liens_obs = explorer(categorie_test, profondeur=1, maximum=10)
fiche = scraper_article(liens_obs[0])
print(f"Page {liens_obs[0]} : {fiche['titre']}")
print(fiche['texte'][:200])


Page /wiki/Artificial_intelligence : Artificial intelligence
Artificial intelligence ( AI ) is the capability of computational systems to perform tasks typically associated with human intelligence , such as learning , reasoning , problem-solving , perception , 


In [134]:
# ✅ Contrôle (ne pas modifier)
controle(fiche is not None and len(fiche['texte']) > 150 and '[' not in fiche['texte'], 'Article scrapé : « %s »' % fiche['titre'], 'Attendu : un dictionnaire avec un texte de plus de 150 caractères, sans renvoi [1]')

✅ Article scrapé : « Artificial intelligence »


## Étape 13/15 — Partie 4 — Le grand lancement : collecter le corpus
On assemble tout : pour chaque thème, on **explore** la catégorie (crawling), puis on **scrape** chaque article. Un `try/except` permet d'**ignorer** un article qui poserait problème au lieu de tout arrêter. ⏱️ Environ **5 minutes** avec la pause d'une seconde : lancez-la, et lisez la suite pendant ce temps.

La cellule utilise **votre** dictionnaire `CATEGORIES`, défini à l'étape « Choisir vos thèmes ». Vous pouvez changer `PAR_CLASSE` (150 donne un corpus plus solide, mais compte une dizaine de minutes).

In [135]:
PAR_CLASSE = 60                                  # textes à garder par thème
lignes = []
for classe, categorie in CATEGORIES.items():
    pris = 0
    for lien in explorer(categorie, profondeur=1, maximum=PAR_CLASSE * 2):
        try:
            fiche = scraper_article(lien)
        except Exception as erreur:              # un article qui pose problème ne doit pas tout arrêter
            print("  ignoré :", lien, "->", erreur)
            continue
        if fiche:
            lignes.append({"classe": classe, **fiche})
            pris += 1
        if pris == PAR_CLASSE:
            break
    print(classe, ":", pris, "textes collectés")
corpus = pd.DataFrame(lignes)
print("Total :", len(corpus), "textes")

IA : 60 textes collectés
Dérive climatique : 60 textes collectés
Voiture hydrogène : 44 textes collectés
Chorégraphes : 60 textes collectés
Total : 224 textes


## Étape 14/15 — Contrôler la qualité, puis sauvegarder
Un corpus se **vérifie** avant de servir : doublons, textes vides, équilibre entre les classes. Un modèle entraîné sur des données sales apprend n'importe quoi.

**À faire :** Dans `corpus` : supprimez les **doublons de titre** et les lignes vides, affichez le **nombre de textes par classe** et la **longueur médiane** des textes (en mots), puis sauvegardez dans **`corpus.csv`** (sans l'index).

**Ce que vous devez obtenir :** le fichier `corpus.csv`, sans doublons, avec l'effectif de chaque thème et la longueur médiane des textes.

> 💡 **Indice** : `drop_duplicates(subset="titre")`, `dropna()`, `value_counts()`, `.str.split().str.len().median()`, `to_csv("corpus.csv", index=False)`.

In [180]:
# TODO : nettoyer corpus, afficher les effectifs et la longueur médiane, sauvegarder
counts = corpus["titre"].value_counts()
print("Titres en doublon :\n",counts[counts > 1])
corpus_clean = corpus.drop_duplicates(subset="titre")
print("Nombre de valeurs na :\n",corpus_clean.isna().sum())
print("Nombre d'articles par catégorie :\n",corpus_clean.classe.value_counts())
print("Nombre de mots médian des textes par classe : \n",corpus_clean.groupby("classe")["texte"].apply(lambda x: x.str.split().str.len().median().astype(int)))
corpus_clean.to_csv("corpus.csv", index=False)

Titres en doublon :
 titre
Mazda Premacy Hydrogen RE Hybrid    3
Name: count, dtype: int64
Nombre de valeurs na :
 classe    0
titre     0
texte     0
lien      0
dtype: int64
Nombre d'articles par catégorie :
 classe
IA                   60
Dérive climatique    60
Chorégraphes         60
Voiture hydrogène    42
Name: count, dtype: int64
Nombre de mots médian des textes par classe : 
 classe
Chorégraphes         53
Dérive climatique    77
IA                   59
Voiture hydrogène    55
Name: texte, dtype: int64


In [181]:
# ✅ Contrôle (ne pas modifier)
relu = pd.read_csv('corpus.csv')
controle(os.path.exists('corpus.csv') and relu['titre'].is_unique and {'texte', 'classe'} <= set(relu.columns) and relu['classe'].nunique() >= 2, 'corpus.csv sauvegardé : %d textes, %d classes' % (len(relu), relu['classe'].nunique()), 'Attendu : un fichier corpus.csv avec colonnes texte et classe, sans doublon de titre, avec au moins 2 classes')

✅ corpus.csv sauvegardé : 222 textes, 4 classes


## Étape 15/15 — API contre scraping : le même texte, obtenu de deux façons
Pour comparer, on récupère le **même article** par l'API (`prop=extracts`) : le texte arrive propre, sans parser de HTML.

In [182]:
titre = corpus["titre"].iloc[0]
donnees = get(BASE + "/w/api.php", {"action": "query", "prop": "extracts", "exintro": 1, "explaintext": 1, "titles": titre, "format": "json"}).json()
extrait = next(iter(donnees["query"]["pages"].values()))["extract"]
print("API      :", len(extrait), "caractères |", extrait[:120])
print("Scraping :", len(corpus["texte"].iloc[0]), "caractères |", corpus["texte"].iloc[0][:120])

API      : 2250 caractères | Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human
Scraping : 504 caractères | Artificial intelligence ( AI ) is the capability of computational systems to perform tasks typically associated with hum


### ✍️ Question
**Comparez les deux méthodes : laquelle est la plus simple, la plus stable, et quand le scraping reste-t-il nécessaire ?**

*Votre réponse (2 à 4 lignes) :*

> 

La méthode API est la plus propre et efficace, dans la mesure où elle est adaptée à une requête par un agent robot. Elle est indépendante de la couche de présentation du texte (html, style, javascript), elle est donc plus fiable et plus stable.

Le scraping est plus lent, car il passe par l'IHM normalement réservée à l'humain, il est donc dépendant de la couche de présentation et est moins stable. Cette méthode est à employer si l'API n'existe pas ou ne répond pas au besoin.

### ✍️ Question
**Votre collecte est-elle « à grande échelle » ? Que faudrait-il changer pour passer de quelques centaines à plusieurs centaines de milliers de textes ?**

*Votre réponse (2 à 4 lignes) :*

> 

224 textes autour des 60 mots n'est pas ce que l'on peut appeler "grande échelle". Pour passer à plusieurs centaines de milliers de textes (x 1000 au moins), l'option scraping parait inenvisageable. Le mieux serait sans doute de répartir la charge des requêtes API sur plusieurs clients (un client se chargeant d'une liste distincte de catégories) qui ne sollicitent pas l'API au même moment.

In [ ]:
# Récupération de tout le texte par l'API (on a enlevé l'option "exintro": 1) en supprimant les lignes et les paragraphes vides :
# il est fort probable qu'un article sera classifié beaucoup plus facilement avec le texte complet qu'avec juste le texte d'introduction
# Dans le texte initial, les paragraphes sont identifiés par '==' ou '===' ou '====' en début de ligne,
# le nombre de '=' correspondant au niveau du paragraphe

class Paragraphe:
    def __init__(self, ligne, parent=None):
        if ligne[:2] != "==":
            # les articles ne commencent pas par un titre de paragraphe mais par un texte simple
            self.niveau = 0
            self.titre = ""
            self.lignes = [ligne]
        else:
            self.niveau = len(ligne.split()[0])
            # le titre correspond au texte compris entre les =
            self.titre = ligne[self.niveau:].split('=')[0]
            # la ligne de titre n'est pas considérée comme ligne de texte, afin de détecter les paragraphes vides
            self.lignes = []
        self.sousParagraphes = []
        self.parent = parent

    def ajoutLigne(self, ligne):
        # cette méthode renvoie l'objet qui va continuer à lire
        # cet objet change si l'on rencontre une ligne commençant par == (fils, parent, frère)
        if ligne[:2] != "==":
            self.lignes.append(ligne)
            return self
        elif len(ligne.split()[0]) > self.niveau:
            # niveau n+1 : on bascule vers un nouveau fils
            nvPar = Paragraphe(ligne,self)
            self.sousParagraphes.append(nvPar)
            return nvPar
        else:
            # le parent (niveau n-1) détermine si cette ligne doit être lue par un nouveau fils frère de cet objet (niveau n),
            # un nouveau frère oncle de cet objet (n-1) ou par son parent grand-père de cet objet (n-2)
            return self.parent.ajoutLigne(ligne)

    def imprime(self):
        # on n'imprime pas les paragraphes sans texte, y compris dans les sous-paragraphes
        if self.nbLignes() == 0:
            return []
        texte = []
        if self.titre != "":
            # on réécrit la ligne de titre car elle n'a pas été enregistrée dans self.lignes
            texte.append('='*self.niveau+' '+self.titre+' '+'='*self.niveau)
        # on commence par le texte du paragraphe
        for ligne in self.lignes:
            texte.append(ligne)
        # on imprime ensuite les sous-paragraphes
        for ssPrg in self.sousParagraphes:
            for ligne in ssPrg.imprime():
                texte.append(ligne)
        return texte

    def nbLignes(self):
        # correspond au nombre de lignes de textes hors titre, y compris celles des sous-paragraphes
        nb = len(self.lignes)
        for ssPrg in self.sousParagraphes:
            nb += ssPrg.nbLignes()
        return nb


indice = 123 # texte au hasard pour tester le bon fonctionnement
titre = corpus["titre"].iloc[indice]
donnees = get(BASE + "/w/api.php", {"action": "query", "prop": "extracts", "explaintext": 1, "titles": titre, "format": "json"}).json()
texte = next(iter(donnees["query"]["pages"].values()))["extract"]
lignes = texte.split("\n")
par = None
for ligne in lignes:
    ligne = ligne.strip()
    if len(ligne) > 0:
        if par == None:
            par = Paragraphe(ligne)
            # on garde l'objet racine pour l'impression globale
            parini = par
        else:
            # l'objet change à chaque nouveau paragraphe ==
            par = par.ajoutLigne(ligne)
texteGlobal = "\n".join(parini.imprime())
print(texteGlobal)


The BMW Hydrogen 7 is a limited production hydrogen internal combustion engine vehicle built from 2005 to 2007 by German automobile manufacturer BMW.  The car is based on BMW's traditional petrol-powered BMW 7 Series (E65) line of vehicles, and more specifically the 760Li.  It uses the same 6-litre V-12 motor as does the 760i and 760Li; however, it has been modified to also allow for the combustion of hydrogen as well as petrol, making it a bivalent engine.
==  Production  ==
Only 100 total vehicles were produced to put their technology to the test.  BMW said it chose public figures such as politicians, media figures, businessmen and big names in the entertainment industry such as 2007 Academy Award-winning director Florian Henckel von Donnersmarck and the chairman of Sixt AG, Erich Sixt, because “they would be ideal ambassadors” for hydrogen fuel and could help spread awareness of the need for such technologies.
==  Use of hydrogen technology  ==
The BMW Hydrogen 7 burns hydrogen in a

In [ ]:
# Par la méthode ci-dessus, on lit tous les articles identifiés précédemment, on stocke le résultat sous corpusTot.csv
total = []
for indice in range(len(corpus_clean)):
    classe = corpus_clean["classe"].iloc[indice]
    titre = corpus_clean["titre"].iloc[indice]
    lien = corpus_clean["lien"].iloc[indice]
    donnees = get(BASE + "/w/api.php", {"action": "query", "prop": "extracts", "explaintext": 1, "titles": titre, "format": "json"}).json()
    texteBrut = next(iter(donnees["query"]["pages"].values()))["extract"]
    lignes = texteBrut.split("\n")
    par = None
    for ligne in lignes:
        ligne = ligne.strip()
        if len(ligne) > 0:
            if par == None:
                par = Paragraphe(ligne)
                parini = par
            else:
                par = par.ajoutLigne(ligne)
    texte = "\n".join(parini.imprime())
    total.append({'classe':classe, 'titre':titre, 'texte':texte, 'lien':lien})
corpusTotal = pd.DataFrame(total)

In [233]:
corpusTotal.to_csv("corpusTot.csv", index=False)